# PARSEC inputs: overview per station

Everything that goes into the parcel runs, per observation station, as it stands in the repo:

1. **Overview table**: hours, period, and where each input comes from (composition tier, cloud-base T/P source, updraft source).
2. **All stations side by side**: distributions of N100, bulk κ, cloud-base temperature and pressure, cloud-base height, σ_w, observed Nd and coupling; median size distributions and compositions.
3. **One page per station**: size distribution, aerosol and droplet number, composition, cloud-base T/P, coupling, updraft, and κ from composition against κ from the CCN counter, with the source of each input per hour.

Reads `inputs/<site>_parcel_inputs.nc` (`build_parcel_inputs.py`, `build_composition.py`, `build_thermo.py`), `updrafts/<site>_w_cloudbase.nc` (`build_updrafts.py`, `zeppelin_bolin.py`) and, where present, `results/<site>_ccn_kappa.nc` (`build_ccn_kappa.py`). Run it from the repo's `notebooks/` folder, or from the repo root.

In [ ]:
import glob
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.patches import Patch

REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, REPO)
from sites import SITE_COORDS, SITES, SPECIES  # noqa: E402

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
REGIME_COL = {"Marine": "#1f77b4", "Boreal": "#2ca02c", "Arctic": "#9467bd", "Continental": "#d62728"}
SPECIES_COL = dict(zip(SPECIES, ["#7f3c8d", "#a5aa99", "#e68310", "#11a579", "#3a3a3a", "#c9a227", "#3969ac", "#80ba5a"]))
print("repo:", REPO)

In [ ]:
def load_site(site):
    """Inputs, updrafts and CCN kappa for one station: an hourly DataFrame, the spectra and the mass fractions."""
    p = os.path.join(REPO, "inputs", f"{site}_parcel_inputs.nc")
    if not os.path.exists(p):
        return None
    with xr.open_dataset(p) as ds:
        ds = ds.load()
    hours = pd.DatetimeIndex(ds["hour"].values)
    df = pd.DataFrame(index=hours)
    for v in ("Nd_obs", "N100", "cloud_base_height", "T0", "P0", "kappa_bulk", "cbh_minus_lcl"):
        df[v] = ds[v].values.astype(float) if v in ds else np.nan
    for v in ("comp_source", "thermo_source"):
        df[v] = ds[v].values.astype(str) if v in ds else "n/a"
    df["T0_C"] = df["T0"] - 273.15
    df["P0_hPa"] = df["P0"] / 100.0
    df["tier"] = df["comp_source"].str.split("+").str[0]
    df["cams_product"] = df["comp_source"].str.extract(r"(cams3h|camsmon|camsclim|nocams)$", expand=False)
    spec = pd.DataFrame(ds["dNdlogDp"].values, index=hours, columns=ds["diameter"].values.astype(float))
    mf = (pd.DataFrame(ds["massfrac"].values, index=hours, columns=[str(s) for s in ds["species"].values])
          if "massfrac" in ds else None)
    dl = np.gradient(np.log10(spec.columns.to_numpy()))
    df["N_total"] = (spec.fillna(0).to_numpy() * dl).sum(axis=1)

    df["sigma_w"], df["w_mean"], df["w_source"] = np.nan, np.nan, "none"
    u = os.path.join(REPO, "updrafts", f"{site}_w_cloudbase.nc")
    if os.path.exists(u):
        with xr.open_dataset(u) as w:
            wh = pd.DatetimeIndex(w["hour"].values)
            df["sigma_w"] = pd.Series(w["sigma_w"].values, index=wh).reindex(hours).values
            wm = w["w_mean_station"] if "w_mean_station" in w else (w["w_mean_obs"] if "w_mean_obs" in w else w["w_mean"])
            df["w_mean"] = pd.Series(wm.values, index=wh).reindex(hours).values
            if "hour_source" in w:
                src = pd.Series(w["hour_source"].values.astype(str), index=wh).reindex(hours).fillna("none")
            else:
                src = pd.Series(np.where(np.isfinite(df["sigma_w"]), str(w.attrs.get("w_source", "file")), "none"),
                                index=hours)
            df["w_source"] = src.values
            for v in ("sigma_w_lidar", "sigma_w_radar"):
                if v in w:
                    df[v] = pd.Series(w[v].values, index=wh).reindex(hours).values

    k = os.path.join(REPO, "results", f"{site}_ccn_kappa.nc")
    if os.path.exists(k):
        with xr.open_dataset(k) as c:
            ss = c["ss"].values.astype(float)
            j = int(np.argmin(np.abs(ss - 0.15)))
            kc = c["kappa_ccn"].where(c["kappa_ok"]).isel(ss=j)
            df["kappa_ccn"] = pd.Series(kc.values, index=pd.DatetimeIndex(c["hour"].values)).reindex(hours).values
            df.attrs["kappa_ccn_ss"] = float(ss[j])
    return dict(site=site, df=df, spec=spec, mf=mf)


SITE_LIST = sorted(os.path.basename(p).replace("_parcel_inputs.nc", "")
                   for p in glob.glob(os.path.join(REPO, "inputs", "*_parcel_inputs.nc")))
DATA = {s: load_site(s) for s in SITE_LIST}
DATA = {s: d for s, d in DATA.items() if d is not None}
print(f"{len(DATA)} stations:", ", ".join(DATA))

## 1. Overview table

In [ ]:
def shares(s, top=4):
    c = pd.Series(s).value_counts()
    return "; ".join(f"{k} {v}" for k, v in c.head(top).items()) + (" …" if len(c) > top else "")


rows = []
for s, d in DATA.items():
    df = d["df"]
    rows.append({
        "station": s, "kind": SITES[s]["kind"], "regime": SITES[s]["regime"],
        "alt (m)": SITE_COORDS[s][2], "hours": len(df), "days": df.index.normalize().nunique(),
        "period": f"{df.index.min():%Y-%m} – {df.index.max():%Y-%m}",
        "Nd obs median": df["Nd_obs"].median(), "N100 median": df["N100"].median(),
        "composition tiers": shares(df["tier"]), "CAMS product": shares(df["cams_product"].fillna("-")),
        "T0/P0 source": shares(df["thermo_source"]), "updraft source": shares(df["w_source"]),
        "σw median": df["sigma_w"].median(),
    })
overview = pd.DataFrame(rows).set_index("station")
overview.style.format({"Nd obs median": "{:.0f}", "N100 median": "{:.0f}", "σw median": "{:.2f}", "alt (m)": "{:.0f}"})

In [ ]:
# share of hours per source: composition tier, cloud-base T/P source, updraft source
counts = {}
for name, col in (("composition tier", "tier"), ("T0/P0 source", "thermo_source"), ("updraft source", "w_source")):
    counts[name] = pd.DataFrame({s: d["df"][col].value_counts() for s, d in DATA.items()}).T.fillna(0).astype(int)
fig, axs = plt.subplots(1, 3, figsize=(18, 0.32 * len(DATA) + 3), sharey=True)
for ax, (name, t) in zip(axs, counts.items()):
    t.div(t.sum(axis=1), axis=0).plot.barh(stacked=True, ax=ax, width=0.8, colormap="tab20", legend=False)
    ax.set(title=name, xlabel="fraction of hours", xlim=(0, 1))
    ax.legend(fontsize=7, loc="lower left", bbox_to_anchor=(0, 1.06), ncol=2, frameon=False)
axs[0].invert_yaxis()
fig.tight_layout()

## 2. All stations side by side

In [ ]:
def site_boxes(col, label, log=False, ax=None):
    order = [s for s in DATA if col in DATA[s]["df"] and DATA[s]["df"][col].notna().any()]
    vals = [DATA[s]["df"][col].dropna().to_numpy() for s in order]
    kw = dict(whis=(10, 90), showfliers=False, patch_artist=True, widths=0.6)
    try:
        bp = ax.boxplot(vals, orientation="horizontal", **kw)       # matplotlib >= 3.10
    except TypeError:
        bp = ax.boxplot(vals, vert=False, **kw)
    for patch, s in zip(bp["boxes"], order):
        patch.set_facecolor(REGIME_COL.get(SITES[s]["regime"], "0.7"))
        patch.set_alpha(0.6)
    ax.set_yticks(range(1, len(order) + 1))
    ax.set_yticklabels(order, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel(label)
    if log:
        ax.set_xscale("log")


panels = [("N100", "N100 (cm$^{-3}$)", True), ("kappa_bulk", "bulk κ (composition)", False),
          ("T0_C", "cloud-base T0 (°C)", False), ("P0_hPa", "cloud-base P0 (hPa)", False),
          ("cloud_base_height", "cloud base above ground (m)", False), ("sigma_w", "σ$_w$ (m s$^{-1}$)", False),
          ("Nd_obs", "observed Nd (cm$^{-3}$)", True), ("cbh_minus_lcl", "cloud base − surface LCL (m)", False)]
fig, axs = plt.subplots(2, 4, figsize=(20, 0.55 * len(DATA) + 3))
for ax, (c, l, lg) in zip(axs.flat, panels):
    site_boxes(c, l, lg, ax)
fig.legend(handles=[Patch(color=c, alpha=0.6, label=k) for k, c in REGIME_COL.items()], ncol=4, loc="upper center",
           title="boxes: interquartile range, line: median, whiskers: 10–90 %", fontsize=9, title_fontsize=9)
fig.tight_layout(rect=(0, 0, 1, 0.93))

In [ ]:
# median size distribution per station (dN/dlogDp), coloured by regime
fig, ax = plt.subplots(figsize=(9, 5.5))
for s, d in DATA.items():
    med = d["spec"].median()
    ax.plot(med.index, med.values, color=REGIME_COL.get(SITES[s]["regime"], "0.5"), lw=1.2, alpha=0.8)
    i = int(np.nanargmax(med.values))
    ax.annotate(s, (med.index[i], med.values[i]), fontsize=7, alpha=0.8)
ax.set(xscale="log", yscale="log", xlabel="diameter (nm)", ylabel="median dN/dlogD$_p$ (cm$^{-3}$)",
       title="median size distribution in the cloud hours")
ax.set_ylim(bottom=1)
ax.legend(handles=[Patch(color=c, label=k) for k, c in REGIME_COL.items()], fontsize=8);

In [ ]:
# median composition per station (mass fractions, renormalised to sum to 1)
mfmed = pd.DataFrame({s: d["mf"].median() for s, d in DATA.items() if d["mf"] is not None}).T
mfmed = mfmed.div(mfmed.sum(axis=1), axis=0)
ax = mfmed.plot.barh(stacked=True, figsize=(10, 0.32 * len(mfmed) + 1.8), width=0.8,
                     color=[SPECIES_COL[c] for c in mfmed.columns])
ax.invert_yaxis()
ax.set(xlabel="median mass fraction", xlim=(0, 1), title="composition used in the parcel runs")
ax.legend(fontsize=8, ncol=4, loc="lower left", bbox_to_anchor=(0, 1.06), frameon=False);

## 3. One page per station

`station_page(site)` draws one station; the last cell draws them all. Hours are plotted in time order by number, since cloud hours are sparse in time.

* **top row**: size distribution (median, 10–90 %); total aerosol, N100 and observed Nd; composition per hour, with the tiers in the title
* **middle row**: cloud-base T0 and P0 per hour coloured by source; coupling (cloud base − surface LCL; shaded: the ±300 m used as "coupled")
* **bottom row**: σ$_w$ per hour coloured by source; lidar against radar σ$_w$ where both exist (else the σ$_w$ distribution); κ from composition against κ from the CCN counter where there is one (else the bulk κ distribution)

In [ ]:
def scatter_by(ax, x, y, cat, title=None):
    cats = pd.Series(np.asarray(cat)).fillna("n/a")
    cmap = plt.get_cmap("tab10")
    for i, c in enumerate(sorted(cats.unique())):
        m = (cats == c).to_numpy()
        ax.plot(x[m], y[m], ".", ms=3, color=cmap(i % 10), label=f"{c} ({m.sum()})")
    ax.legend(fontsize=7, markerscale=3, loc="best")
    if title:
        ax.set_title(title, fontsize=10)


def station_page(site):
    d = DATA[site]
    df, spec, mf = d["df"], d["spec"], d["mf"]
    t = np.arange(len(df))
    fig, axs = plt.subplots(3, 3, figsize=(17, 12))
    fig.suptitle(f"{site} — {SITES[site]['kind']}, {SITES[site]['regime']}, {SITE_COORDS[site][2]:.0f} m a.s.l.; "
                 f"{len(df)} cloud hours on {df.index.normalize().nunique()} days "
                 f"({df.index.min():%Y-%m-%d} – {df.index.max():%Y-%m-%d})", fontsize=12)

    ax = axs[0, 0]
    q = spec.quantile([0.1, 0.5, 0.9])
    ax.fill_between(spec.columns, q.loc[0.1], q.loc[0.9], color="0.8", label="10–90 %")
    ax.plot(spec.columns, q.loc[0.5], "k", label="median")
    lo = np.nanmin(q.loc[0.1].where(q.loc[0.1] > 0))
    ax.set(xscale="log", yscale="log", xlabel="diameter (nm)", ylabel="dN/dlogD$_p$ (cm$^{-3}$)",
           title="size distribution", ylim=(max(0.1, lo / 2) if np.isfinite(lo) else 0.1, None))
    ax.legend(fontsize=8)

    ax = axs[0, 1]
    for col, c, lab in (("N_total", "0.5", "N total"), ("N100", "#d62728", "N100"), ("Nd_obs", "#1f77b4", "Nd observed")):
        ax.plot(t, df[col], ".", ms=2, color=c, label=lab)
    ax.set(yscale="log", xlabel="cloud hour (time order)", ylabel="cm$^{-3}$", title="aerosol and droplet number")
    ax.legend(fontsize=8, markerscale=4)

    ax = axs[0, 2]
    if mf is not None:
        m = mf.div(mf.sum(axis=1), axis=0).fillna(0)
        ax.stackplot(t, m.T.to_numpy(), colors=[SPECIES_COL[c] for c in m.columns], labels=m.columns, lw=0)
        ax.set(xlim=(0, max(len(t) - 1, 1)), ylim=(0, 1), xlabel="cloud hour (time order)", ylabel="mass fraction")
        ax.set_title("composition: " + shares(df["tier"]), fontsize=10)
        ax.legend(fontsize=6, ncol=4, loc="lower left")

    ax = axs[1, 0]
    scatter_by(ax, t, df["T0_C"].to_numpy(), df["thermo_source"], "cloud-base T0 by source")
    ax.axhline(0, color="k", lw=0.6)
    ax.axhline(-10, color="k", lw=0.6, ls=":")
    ax.set(xlabel="cloud hour (time order)", ylabel="T0 (°C)")

    ax = axs[1, 1]
    scatter_by(ax, t, df["P0_hPa"].to_numpy(), df["thermo_source"], "cloud-base P0 by source")
    ax.set(xlabel="cloud hour (time order)", ylabel="P0 (hPa)")
    ax.invert_yaxis()

    ax = axs[1, 2]
    x = df["cbh_minus_lcl"].dropna()
    if len(x):
        ax.hist(x.clip(-1000, 2000), bins=np.arange(-1000, 2001, 100), color="0.5")
        ax.axvspan(-300, 300, color="#2ca02c", alpha=0.15, label=f"coupled (|d| ≤ 300 m): {(x.abs() <= 300).mean():.0%}")
        ax.set(xlabel="cloud base − surface LCL (m)", ylabel="hours")
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, "no surface met, or in-cloud station", ha="center", va="center", transform=ax.transAxes)
    ax.set_title("coupling", fontsize=10)

    ax = axs[2, 0]
    if df["sigma_w"].notna().any():
        scatter_by(ax, t, df["sigma_w"].to_numpy(), df["w_source"], "σ$_w$ by source")
        ax.set(xlabel="cloud hour (time order)", ylabel="σ$_w$ (m s$^{-1}$)")
    else:
        ax.text(0.5, 0.5, "no updraft file", ha="center", va="center", transform=ax.transAxes)

    ax = axs[2, 1]
    both = df[[c for c in ("sigma_w_lidar", "sigma_w_radar") if c in df]].dropna()
    if both.shape[1] == 2 and len(both) > 5:
        ax.plot(both["sigma_w_lidar"], both["sigma_w_radar"], ".", ms=4)
        lim = [0, max(1.5, float(both.max().max()))]
        ax.plot(lim, lim, "k-", lw=0.8)
        ax.set(xlim=lim, ylim=lim, xlabel="σ$_w$ lidar, below base", ylabel="σ$_w$ radar, above base (unscaled)")
        ax.set_title(f"lidar vs radar ({len(both)} h)", fontsize=10)
    elif df["sigma_w"].notna().any():
        ax.hist(df["sigma_w"].dropna(), bins=np.arange(0, 2.01, 0.05), color="0.5")
        title = "σ$_w$ distribution"
        if df["w_mean"].notna().any() and abs(df["w_mean"].median()) > 0.05:
            title += f" (station mean w, median {df['w_mean'].median():+.2f} m/s)"
        ax.set(xlabel="σ$_w$ (m s$^{-1}$)", ylabel="hours")
        ax.set_title(title, fontsize=10)

    ax = axs[2, 2]
    if "kappa_ccn" in df and df["kappa_ccn"].notna().sum() > 5:
        ok = df[["kappa_bulk", "kappa_ccn"]].dropna()
        ax.plot(ok["kappa_bulk"], ok["kappa_ccn"], ".", ms=4)
        ax.plot([0, 1.2], [0, 1.2], "k-", lw=0.8)
        ax.set(xlim=(0, 1.2), ylim=(0, 1.2), xlabel="bulk κ (composition)",
               ylabel=f"κ from CCN counter at {df.attrs.get('kappa_ccn_ss', np.nan):.2f} %")
        ax.set_title(f"composition vs CCN κ ({len(ok)} h)", fontsize=10)
    else:
        ax.hist(df["kappa_bulk"].dropna(), bins=np.arange(0, 1.21, 0.025), color="0.5")
        ax.set(xlabel="bulk κ (composition)", ylabel="hours")
        ax.set_title("bulk κ (no CCN counter)", fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    return fig

In [ ]:
# one station
station_page("hyytiala" if "hyytiala" in DATA else next(iter(DATA)));

In [ ]:
# all stations; SAVE = True also writes figures/inputs_overview/<site>.png
SAVE = False
for s in DATA:
    fig = station_page(s)
    if SAVE:
        os.makedirs(os.path.join(REPO, "figures", "inputs_overview"), exist_ok=True)
        fig.savefig(os.path.join(REPO, "figures", "inputs_overview", f"{s}.png"), dpi=120)
    plt.show()